In [2]:
from datetime import date
import polars as pl
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
df_flights = get_sfdps_flights_day(date(2026,3,1))

In [5]:
df_flights.filter(pl.col("takeoff_airport_ident") == "MC1")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a0eb2c""","""N15862""","""N15862""",2026-03-01 18:04:00,"""MC1""",2026-03-01 18:45:00,"""PAO""",false,false,false,false,"""""","""""","""""",""""""
"""a22233""","""CXK312""",null,2026-03-01 19:29:00,"""MC1""",2026-03-01 20:43:00,"""KSAC""",false,false,false,false,"""""","""""","""""",""""""
"""a50369""","""N422DS""","""N422DS""",2026-03-01 03:50:00,"""MC1""",2026-03-01 04:45:00,"""FAT""",false,false,false,false,"""""","""""","""""",""""""
"""a683f3""","""N519MB""","""N519MB""",2026-03-01 17:12:00,"""MC1""",2026-03-01 18:21:00,"""SAC""",false,false,false,false,"""""","""""","""""",""""""
"""a72112""","""N559CF""","""N559CF""",2026-03-01 23:00:00,"""MC1""",2026-03-01 23:52:00,"""FAT""",false,false,false,false,"""""","""""","""""",""""""
"""a749d9""","""CXK448""",null,2026-03-01 01:19:00,"""MC1""",2026-03-01 02:10:00,"""SAC""",false,false,false,false,"""""","""""","""""",""""""
"""a8b372""","""N66SB""","""N66SB""",2026-03-01 02:20:00,"""MC1""",2026-03-01 02:54:00,"""SQL""",false,false,false,false,"""""","""""","""""",""""""
"""acd3a4""","""CXK609""",null,2026-03-01 03:02:00,"""MC1""",2026-03-01 03:49:00,"""SAC""",false,false,false,false,"""""","""""","""""",""""""
"""ad2d78""","""N94814""","""N94814""",2026-03-01 21:09:00,"""MC1""",2026-03-01 21:38:00,"""WVI""",false,false,false,false,"""""","""""","""""",""""""


In [13]:
df_flights.filter(pl.col("takeoff_airport_ident").str.len_chars() == 3)
# check if just 3 letters

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a00d18""","""N102KW""","""N102KW""",2026-03-01 19:00:00,"""0W3""",2026-03-01 19:32:00,"""KESN""",false,false,false,false,"""""","""""","""""",""""""
"""a0117d""","""N103SV""","""N103SV""",2026-03-01 21:31:00,"""3T5""",2026-03-01 23:58:00,"""KBRO""",false,false,false,false,"""""","""""","""""",""""""
"""a01cda""","""N106VA""","""N106VA""",2026-03-01 05:03:00,"""SC1""",2026-03-01 06:59:00,"""KLVK""",false,false,false,false,"""""","""""","""""",""""""
"""a02209""","""N1077G""","""N1077G""",2026-03-01 22:26:00,"""26N""",2026-03-01 23:11:00,"""39N""",false,false,false,false,"""""","""""","""""",""""""
"""a03fb8""","""KSF15""",null,2026-03-01 01:59:00,"""1G3""",2026-03-01 02:54:00,"""AGC""",false,false,false,false,"""""","""""","""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adeccf""","""N997BT""","""N997BT""",2026-03-01 16:47:00,"""Y51""",2026-03-01 17:42:00,"""KUES""",false,false,false,false,"""""","""""","""""",""""""
"""adf4ad""","""N999GE""","""N999GE""",2026-03-01 19:18:00,"""CVA""",2026-03-01 19:41:00,"""C17""",false,false,false,false,"""""","""""","""""",""""""
"""adf6a1""","""N9991V""","""N9991V""",2026-03-01 20:39:00,"""M98""",2026-03-01 22:27:00,"""GRB""",false,false,false,false,"""""","""""","""""",""""""


In [ ]:
# see which whether it be ident, iata, gps_code, local_code is most common match in sfdps and then do that as the first phoen to get them all on ident. 

In [14]:
from airports.airportLookup import AirportLookup
from IPython.display import display


airport_lookup = AirportLookup()

airport_indexes = {
    "ident_index": airport_lookup.ident_index,
    "iata_to_ident_index": airport_lookup.iata_to_ident_index,
    "gps_code_to_ident_index": airport_lookup.gps_code_to_ident_index,
    "local_code_to_ident_index": airport_lookup.local_code_to_ident_index,
}


def index_match_summary(airport_column: str) -> pl.DataFrame:
    codes = df_flights.get_column(airport_column).drop_nulls()
    total_rows = df_flights.height
    non_null_rows = len(codes)
    rows = []

    for index_name, airport_index in airport_indexes.items():
        mapped_rows = codes.is_in(set(airport_index)).sum()
        rows.append(
            {
                "airport_column": airport_column,
                "index": index_name,
                "mapped_rows": mapped_rows,
                "unmapped_non_null_rows": non_null_rows - mapped_rows,
                "non_null_rows": non_null_rows,
                "total_rows": total_rows,
                "mapped_pct": mapped_rows / non_null_rows if non_null_rows else 0,
            }
        )

    return pl.DataFrame(rows).sort(
        ["airport_column", "mapped_rows"],
        descending=[False, True],
    )


airport_index_summary = pl.concat(
    [
        index_match_summary("takeoff_airport_ident"),
        index_match_summary("landing_airport_ident"),
    ]
)

best_airport_indexes = (
    airport_index_summary.sort(
        ["airport_column", "mapped_rows"],
        descending=[False, True],
    )
    .group_by("airport_column", maintain_order=True)
    .first()
)

display(airport_index_summary)
display(best_airport_indexes)

airport_column,index,mapped_rows,unmapped_non_null_rows,non_null_rows,total_rows,mapped_pct
str,str,i64,i64,i64,i64,f64
"""takeoff_airport_ident""","""gps_code_to_ident_index""",26279,529,26808,26808,0.980267
"""takeoff_airport_ident""","""ident_index""",26166,642,26808,26808,0.976052
"""takeoff_airport_ident""","""local_code_to_ident_index""",298,26510,26808,26808,0.011116
"""takeoff_airport_ident""","""iata_to_ident_index""",59,26749,26808,26808,0.002201
"""landing_airport_ident""","""gps_code_to_ident_index""",25159,1649,26808,26808,0.938489
"""landing_airport_ident""","""ident_index""",25076,1732,26808,26808,0.935392
"""landing_airport_ident""","""local_code_to_ident_index""",1712,25096,26808,26808,0.063862
"""landing_airport_ident""","""iata_to_ident_index""",1144,25664,26808,26808,0.042674


airport_column,index,mapped_rows,unmapped_non_null_rows,non_null_rows,total_rows,mapped_pct
str,str,i64,i64,i64,i64,f64
"""landing_airport_ident""","""gps_code_to_ident_index""",25159,1649,26808,26808,0.938489
"""takeoff_airport_ident""","""gps_code_to_ident_index""",26279,529,26808,26808,0.980267


In [20]:

iata_index = set(airport_lookup.iata_to_ident_index)
gps_code_index = set(airport_lookup.gps_code_to_ident_index)
local_code_index = set(airport_lookup.local_code_to_ident_index)
ident_index = set(airport_lookup.ident_index)

In [23]:
# should probalby use not ident but each of the iata and gps. 
df_flights.filter(~pl.col("takeoff_airport_ident").is_in(iata_index) & ~pl.col("takeoff_airport_ident").is_in(gps_code_index) & ~pl.col("takeoff_airport_ident").is_in(local_code_index))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a010c0""","""N103KF""","""N103KF""",2026-03-01 18:48:00,"""POISN305024""",2026-03-01 20:38:00,"""KTVY""",false,false,false,false,"""""","""""","""""",""""""
"""a01cda""","""N106VA""","""N106VA""",2026-03-01 05:03:00,"""SC1""",2026-03-01 06:59:00,"""KLVK""",false,false,false,false,"""""","""""","""""",""""""
"""a022d3""","""N108EC""","""N108EC""",2026-03-01 12:46:00,"""PALEO""",2026-03-01 13:22:00,"""PALEO""",false,false,false,false,"""""","""""","""""",""""""
"""a032d8""","""N1114K""","""N1114K""",2026-03-01 02:43:00,"""RBL125027""",2026-03-01 04:13:00,"""KMOD""",false,false,false,false,"""""","""""","""""",""""""
"""a0463e""","""N117AK""","""N117AK""",2026-03-01 14:11:00,"""KLFK""",2026-03-01 16:07:00,"""H92""",false,false,false,false,"""""","""""","""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ad890d""","""N971TH""","""N971TH""",2026-03-01 14:12:00,"""NMM225004""",2026-03-01 16:47:00,"""KJFX""",false,false,false,false,"""""","""""","""""",""""""
"""adbb18""","""N984ME""","""N984ME""",2026-03-01 06:31:00,"""ZZ6""",2026-03-01 07:19:00,"""FAT""",false,false,false,false,"""""","""""","""""",""""""
"""add907""","""N9912Q""","""N9912Q""",2026-03-01 02:34:00,"""SC3""",2026-03-01 03:09:00,"""F70""",false,false,false,false,"""""","""""","""""",""""""
